# Data collection and filtering approach

This notebook produces every number used in the *Data collection and filtering approach* subsection of the project-health part of the report. Each part below starts with the text from the report, followed by the code and output that back it up.

We analyse calendar year 2025 of `kubernetes/kubernetes`, using earlier years as baseline.

In [1]:
import os, re, subprocess, unicodedata
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
pd.set_option("display.max_colwidth", 100)

In [2]:
# constants

k8_url = "https://github.com/kubernetes/kubernetes"
repo_path = "repo.git"
start = pd.Timestamp("2025-01-01", tz="UTC")
end = pd.Timestamp("2026-01-01", tz="UTC")
os.makedirs("data", exist_ok=True)

In [3]:
# clone only the commit history (no file contents), takes a few seconds

if not os.path.exists(repo_path):
    subprocess.run(["git", "clone", "--bare", "--filter=tree:0", k8_url, repo_path], check=True)

def git(*args):
    return subprocess.run(["git", "-C", repo_path, *args], capture_output=True,
                          text=True, encoding="utf-8", errors="replace").stdout

print("commits on master:", git("rev-list", "--count", "master").strip())

commits on master: 142253


In [4]:
# export every commit on master: sha, parents, author, dates, message

fmt = "%x1e%H%x1f%P%x1f%an%x1f%ae%x1f%aI%x1f%cI%x1f%s%x1f%b"
raw = git("log", "master", f"--format={fmt}")

rows = []
for rec in raw.split("\x1e"):
    parts = rec.split("\x1f")
    if len(parts) < 8:
        continue
    sha, parents, name, email, author_date, commit_date, subject, body = parts[:8]
    rows.append({"sha": sha.strip(), "parents": parents, "author_name": name,
                 "author_email": email.lower(), "author_date": author_date,
                 "commit_date": commit_date, "subject": subject, "body": body.strip()})

commits = pd.DataFrame(rows)
commits["author_date"] = pd.to_datetime(commits.author_date, utc=True)
commits["commit_date"] = pd.to_datetime(commits.commit_date, utc=True)
commits["is_merge"] = commits.parents.str.split().str.len() > 1
print(len(commits), "commits")

142253 commits


In [5]:
# sanity check against the interim report (search API said 7141 commits in 2025)

c25 = commits[(commits.commit_date >= start) & (commits.commit_date < end)]
print("2025 commits:", len(c25))
print("of which PR merge commits:", c25.is_merge.sum())

2025 commits: 7141
of which PR merge commits: 2612


In [6]:
# every merge commit names the PR number and the fork (login) it came from

merge_re = r"^Merge pull request #(\d+) from ([^/\s]+)/(\S+)"
found = commits.subject.str.extract(merge_re)
commits["pr_number"] = pd.to_numeric(found[0])
commits["pr_login"] = found[1]

prs = commits[commits.pr_number.notna()].copy()
prs = prs.rename(columns={"commit_date": "merged_at"})
prs["pr_number"] = prs.pr_number.astype(int)
prs["pr_title"] = prs.body.str.split("\n").str[0]
prs = prs[["pr_number", "pr_login", "pr_title", "merged_at", "sha", "parents"]]
prs = prs.sort_values("merged_at").drop_duplicates("pr_number")

prs25 = prs[(prs.merged_at >= start) & (prs.merged_at < end)]
print("merged PRs all time:", len(prs), "| in 2025:", len(prs25))
prs25.head(10)

merged PRs all time: 58206 | in 2025: 2612


,pr_number,pr_login,pr_title,merged_at,sha,parents
14763,129119,macsko,Fix PodTopologySpread matching pods counts for constraints with the same topologyKey,2025-01-01 10:04:14+00:00,1c2b2cce10eb8fb136e16bdbcff6d45990dcdd2a,d8d369128012ef5651f0af67620a6e7f85627771 c3a54926a4412db35f8f3e2d295832aace0651fa
14762,129401,pacoxu,kubeadm: add kernel version check for upgrade,2025-01-02 02:50:13+00:00,3c229949f992d9f798bcfd9f79dab88c21500c36,1c2b2cce10eb8fb136e16bdbcff6d45990dcdd2a 2c305d71f0516e596906d1f03ffef991a786ba1a
14759,129452,SataQiu,kubeadm: fix a bug where the 'node.skipPhases' in UpgradeConfiguration is not respected by 'kube...,2025-01-02 11:00:14+00:00,87cf098f34b6782d20cbfaced2b4260a595ac066,3c229949f992d9f798bcfd9f79dab88c21500c36 4c6deed07ecd31dc1cd7f558f241e9354e7ccbdf
14758,129419,googs1025,chore(printers): add miss unit test for resourcequota,2025-01-02 12:12:14+00:00,281bba84aba419049b287e3afd81d97a9af9e8fa,87cf098f34b6782d20cbfaced2b4260a595ac066 b649c1136614d147e56ffef217ac714810e6a9b2
14755,127153,Anurag252,Add a static NodePort allocator for e2e tests,2025-01-02 15:56:14+00:00,6746df77f2376c6bc1fd0de767d2a94e6bd6cec1,281bba84aba419049b287e3afd81d97a9af9e8fa a9b641464ebcb2179f04450180802e0994cc51fe
14749,128872,alvaroaleman,Use generic btree in watchcache,2025-01-03 13:42:14+00:00,8f8c94a04d00e59d286fe4387197bc62c6a4f374,6746df77f2376c6bc1fd0de767d2a94e6bd6cec1 b1cb471982b74c13c26dbcc0f4e1b5ae92ea47e6
14747,128972,manishym,Add crd from external snapshotter,2025-01-06 10:24:16+00:00,b358cf3a791bce0df23a52bb32b0e47e23746a8d,8f8c94a04d00e59d286fe4387197bc62c6a4f374 446252b0c359b6d6764d91bdf018ff4bea561fe0
14744,129349,dims,Bump x/net to v0.33.0,2025-01-06 19:04:38+00:00,b7ef173c59065f9a5f68eb514ef0483c6f3887ae,7c887412c882afc130d78e4bb6efad056b0fe376 0b6e3718340fa7e3846cf9b7d5a0f7a684a6fa5a
14743,129430,MadhavJivrajani,[go1.24] webhook: alter regex to account for x509sha1 GODEBUG removal,2025-01-06 20:28:30+00:00,c3f3fdc1aa62002a58bec1141fe69e86bbb27491,b7ef173c59065f9a5f68eb514ef0483c6f3887ae cff0f4009f17c84141553c143872d47756209f4d
14740,128256,lzhecheng,[e2e][webhook] Retry update ValidatingWebhookConfigurations,2025-01-07 04:06:29+00:00,3a79d453fa4126ce3acb87fd1061881e736f6239,c3f3fdc1aa62002a58bec1141fe69e86bbb27491 49b479116f4c4f58ca33d11933509bbc8ab6b087


In [7]:
# which commits did each PR bring in? (everything between the two parents of its merge commit)

def pr_commit_list(row):
    p = row.parents.split()
    shas = git("rev-list", "--no-merges", f"{p[0]}..{p[1]}").split()
    return [(row.pr_number, s) for s in shas]

links = []
with ThreadPoolExecutor(16) as ex:
    for res in ex.map(pr_commit_list, prs.itertuples(), chunksize=64):
        links.extend(res)

pr_commits = pd.DataFrame(links, columns=["pr_number", "sha"])
print(len(pr_commits), "PR-commit links for", pr_commits.pr_number.nunique(), "PRs")

82631 PR-commit links for 58202 PRs


# Bots


In [8]:
# measure 1 (name): name / email / login looks like a bot

known_bots = {"k8s-ci-robot", "k8s-merge-robot", "k8s-release-robot", "k8s-triage-robot",
              "fejta-bot", "k8s-github-robot", "dependabot", "dependabot[bot]", "k8s-bot", "googlebot"}
bot_re = re.compile(r"(\[bot\]|-bot\b|-robot\b|\brobot\b|\bbot\b)", re.I)

def name_measure(name, email):
    local = re.sub(r"^\d+\+", "", email.split("@")[0])
    return bool(bot_re.search(name) or bot_re.search(local) or local in known_bots)

# measure 2 (behaviour): same message over and over, never through a PR

def template(subject):
    # mask version numbers so "bump to v1.2" and "bump to v1.3" count as the same message
    s = re.sub(r"v?\d+(\.\d+)*(-[a-z]+\.\d+)?", "#", subject.lower())
    return re.sub(r"\s+", " ", s).strip()

commits["template"] = commits.subject.map(template)
commits["via_pr"] = commits.sha.isin(set(pr_commits.sha))

emails = commits.groupby("author_email").agg(
    names=("author_name", lambda s: sorted(set(s))), n=("sha", "size"),
    n_merge=("is_merge", "sum"), n_templates=("template", "nunique"),
    n_via_pr=("via_pr", "sum"), first=("author_date", "min"), last=("author_date", "max"))
emails["template_ratio"] = emails.n_templates / emails.n
emails["share_via_pr"] = emails.n_via_pr / (emails.n - emails.n_merge).clip(lower=1)

emails["name_measure"] = [any(name_measure(nm, e) for nm in names) for e, names in zip(emails.index, emails.names)]
emails["behaviour_measure"] = (emails.n >= 10) & (emails.template_ratio < 0.25) & (emails.share_via_pr == 0)
emails["is_bot"] = emails.name_measure | emails.behaviour_measure

emails[emails.is_bot][["names", "n", "n_templates", "name_measure", "behaviour_measure"]]

,names,n,n_templates,name_measure,behaviour_measure
author_email,,,,,
181008794+kubernetes-prow[bot]@users.noreply.github.com,[kubernetes-prow[bot]],1269,1232,True,False
20407524+k8s-ci-robot@users.noreply.github.com,[Kubernetes Prow Robot],6599,6226,True,False
k8s-ci-robot@users.noreply.github.com,"[Kubernetes Prow Robot, k8s-ci-robot]",20726,19057,True,False
k8s-merge-robot@users.noreply.github.com,[Kubernetes Submit Queue],15244,14106,True,False
k8s-release-robot@users.noreply.github.com,[Kubernetes Release Robot],313,1,True,True
k8s.production.user@gmail.com,[k8s-merge-robot],4953,4543,True,False
nobody@k8s.io,[Anago GCB],320,7,False,True


In [9]:
# k8s-ci-robot: share of all 2025 commits

bot_emails = set(emails[emails.is_bot].index)
c25_bots = c25[c25.author_email.isin(bot_emails)]
prow = c25_bots[c25_bots.author_name == "Kubernetes Prow Robot"]
print("Prow commits 2025:", len(prow), f"= {len(prow)/len(c25):.1%} of all 2025 commits")
print("all of them merge commits:", prow.is_merge.all())
print("all 2025 bot commits:", c25_bots.author_name.value_counts().to_dict())

Prow commits 2025: 2612 = 36.6% of all 2025 commits
all of them merge commits: True
all 2025 bot commits: {'Kubernetes Prow Robot': 2612, 'Kubernetes Release Robot': 62}


In [10]:
# k8s-release-robot: same message every time, never through a PR

rel = c25[c25.author_name == "Kubernetes Release Robot"]
print(len(rel), "commits in 2025")
print("all start with 'CHANGELOG: Update directory for':", rel.subject.str.startswith("CHANGELOG: Update directory for").all())
print("any of them through a PR:", rel.sha.isin(pr_commits.sha).any())
rel.subject.head(5)

62 commits in 2025
all start with 'CHANGELOG: Update directory for': True
any of them through a PR: False


7843         CHANGELOG: Update directory for v1.35.0 release
7845        CHANGELOG: Update directory for v1.32.11 release
7889         CHANGELOG: Update directory for v1.33.7 release
7890         CHANGELOG: Update directory for v1.34.3 release
7891    CHANGELOG: Update directory for v1.35.0-rc.1 release
Name: subject, dtype: str

In [11]:
# Anago GCB: no bot-like name, caught only by the behaviour measure

emails.loc[["nobody@k8s.io"], ["names", "n", "n_templates", "share_via_pr", "name_measure", "behaviour_measure"]]

,names,n,n_templates,share_via_pr,name_measure,behaviour_measure
author_email,,,,,,
nobody@k8s.io,[Anago GCB],320,7,0.0,False,True


# Multiple accounts per person


In [12]:
# every commit email and every PR login is a node; linking two nodes means "same person"
# (union-find: find() returns the group a node belongs to, union() joins two groups)

parent = {}

def find(x):
    parent.setdefault(x, x)
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

link_log = []
def union(a, b, why):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[ra] = rb
        link_log.append((a, b, why))

humans = emails[~emails.is_bot]
login_counts = prs.pr_login.str.lower().value_counts()
login_is_bot = {l: bool(bot_re.search(l)) or l in known_bots for l in login_counts.index}

for e in humans.index:
    find("email:" + e)                 # step 1: names on one email are already one node
for l in login_counts.index:
    if not login_is_bot[l]:
        find("login:" + l)

In [13]:
# measure 1: name and email

def name_key(s):
    t = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode().lower()
    t = re.sub(r"[^a-z]", "", t)
    if len(t) >= 8 and len(s.split()) >= 2:                       # full name, e.g. "Marek Siarkowicz"
        return "n:" + t
    if len(t) >= 7 and len(s.split()) == 1 and re.search(r"[A-Z].*[A-Z]|\d", s):
        return "n:" + t                                             # distinctive handle, e.g. "SataQiu"
    if re.search(r"[\u4e00-\u9fff]", s) and len(s.strip()) >= 2:   # chinese name
        return "c:" + s.strip()
    return None

generic = {"admin", "root", "dev", "info", "user", "test", "mail", "github", "contact", "me", "hello", "noreply"}
by_name, by_prefix = defaultdict(set), defaultdict(set)

for e, row in humans.iterrows():
    prefix = e.split("@")[0]
    m = re.match(r"^\d+\+(.+)$", prefix)
    if m and e.endswith("users.noreply.github.com"):          # 123+login@users.noreply.github.com
        union("email:" + e, "login:" + m.group(1).lower(), "measure 1: noreply email contains login")
        prefix = m.group(1).lower()
    if prefix not in generic and len(prefix) >= 6 and (len(prefix) >= 9 or re.search(r"[._\-\d]", prefix)):
        by_prefix[prefix].add(e)
    for nm in row.names:
        k = name_key(nm)
        if k:
            by_name[k].add(e)

for group in by_name.values():
    group = sorted(group)
    for e in group[1:]:
        union("email:" + group[0], "email:" + e, "measure 1: same name")
for group in by_prefix.values():
    group = sorted(group)
    for e in group[1:]:
        union("email:" + group[0], "email:" + e, "measure 1: same email prefix")

print(pd.Series([w for _, _, w in link_log]).value_counts())

measure 1: same name                       563
measure 1: noreply email contains login    234
measure 1: same email prefix                20
Name: count, dtype: int64


In [14]:
# measure 2: activity -- in whose PRs do an email's commits end up?

x = pr_commits.merge(commits[["sha", "author_email"]], on="sha").merge(prs[["pr_number", "pr_login"]], on="pr_number")
x = x[x.author_email.isin(humans.index)]
x["pr_login"] = x.pr_login.str.lower()
cnt = x.groupby(["author_email", "pr_login"]).size().rename("k").reset_index()
cnt["share"] = cnt.k / cnt.groupby("author_email").k.transform("sum")

def similar(login, email):
    # login shares 4 letters in a row with the email prefix or an author name
    lg = re.sub(r"[^a-z]", "", login.lower())
    cands = [re.sub(r"[^a-z]", "", email.split("@")[0].lower())]
    cands += [re.sub(r"[^a-z]", "", unicodedata.normalize("NFKD", n).encode("ascii", "ignore").decode().lower())
              for n in humans.names.get(email, [])]
    return any(len(c) >= 4 and lg[i:i + 4] in c for c in cands for i in range(len(lg) - 3))

cnt["similar"] = [similar(l, e) for l, e in zip(cnt.pr_login, cnt.author_email)]
# most of its commits (> 50%) AND (at least 3 commits OR login resembles the name)
strong = cnt[(cnt.share > 0.5) & ((cnt.k >= 3) | cnt.similar)]
print(len(strong), "email -> login links")

4575 email -> login links


In [15]:
# logins used by several people are never used for linking (explained in the next part)

per_login = defaultdict(lambda: defaultdict(int))
for r in strong.itertuples():
    per_login[r.pr_login][find("email:" + r.author_email)] += r.k
login_emails = strong.groupby("pr_login").author_email.apply(list).to_dict()

def is_shared(login, people):
    ks = sorted(people.values(), reverse=True)
    several = len(ks) >= 2 and ks[1] >= 2 and ks[1] >= 0.2 * ks[0]   # 2+ people, nobody dominates
    return several and not any(similar(login, e) for e in login_emails[login])

shared = {l: dict(v) for l, v in per_login.items() if is_shared(l, v)}

for r in strong.itertuples():
    if not login_is_bot.get(r.pr_login, False) and r.pr_login not in shared:
        union("email:" + r.author_email, "login:" + r.pr_login, "measure 2: activity")

print(pd.Series([w for _, _, w in link_log]).value_counts())

measure 2: activity                        3913
measure 1: same name                        563
measure 1: noreply email contains login     234
measure 1: same email prefix                 20
Name: count, dtype: int64


In [16]:
# example: SataQiu -- which measure made which link?

sata = [l for l in link_log if any(k in (l[0] + l[1]).lower() for k in ["sataqiu", "shidaqiu", "qiushida", "1527062125"])]
for a, b, why in sata:
    print(f"{a:40s} <-> {b:35s} {why}")
print()
print(commits[commits.author_email.isin(["1527062125@qq.com", "qiushida@beyondcent.com", "shidaqiu2018@gmail.com"])]
      .groupby(["author_email", "author_name"]).size())

email:1527062125@qq.com                  <-> email:qiushida@beyondcent.com       measure 1: same name
email:1527062125@qq.com                  <-> email:shidaqiu2018@gmail.com        measure 1: same name
email:1527062125@qq.com                  <-> login:sataqiu                       measure 2: activity

author_email             author_name
1527062125@qq.com        SataQiu         90
qiushida@beyondcent.com  SataQiu        184
shidaqiu2018@gmail.com   SataQiu        220
                         Shida Qiu        1
dtype: int64


In [17]:
# how many people were active in 2025?

person = {n: find(n) for n in list(parent)}
ids = {root: i for i, root in enumerate(sorted(set(person.values())))}
person_map = pd.DataFrame({"node": list(person), "person_id": [ids[r] for r in person.values()]})
pid = dict(zip(person_map.node, person_map.person_id))

emails25 = set(c25.author_email) - bot_emails
logins25 = {l for l in set(prs25.pr_login.str.lower()) if not login_is_bot.get(l)}
nodes25 = {"email:" + e for e in emails25} | {"login:" + l for l in logins25 if l not in shared}
print("commit emails active in 2025:", len(emails25))
print("PR logins active in 2025:", len(logins25))
print("people with attributable work in 2025:", len({pid[n] for n in nodes25 if n in pid}))

commit emails active in 2025: 526
PR logins active in 2025: 466
people with attributable work in 2025: 553


In [18]:
# manual check: 20 random people active in 2025 that consist of >= 2 nodes

multi = person_map.groupby("person_id").node.apply(list)
multi = multi[(multi.str.len() >= 2) & multi.index.isin({pid[n] for n in nodes25 if n in pid})]
sample = multi.sample(20, random_state=42)

check = []
for p_id, members in sample.items():
    names = sorted({nm for m in members if m.startswith("email:") for nm in humans.names.get(m[6:], [])})
    check.append({"logins": [m[6:] for m in members if m.startswith("login:")],
                  "names": names, "emails": [m[6:] for m in members if m.startswith("email:")]})
check = pd.DataFrame(check)
check.to_csv("data/identity_validation_sample.csv", index=False)
check

,logins,names,emails
0,[narita-1095305],[Rikiya Narita],[b2560305@planet.kanazawa-it.ac.jp]
1,[serathius],[Marek Siarkowicz],"[marek.siarkowicz@protonmail.com, serathius@users.noreply.github.com, siarkowicz@google.com]"
2,[devmanishoffl],[devmanishofficial],[devmanishofficial@gmail.com]
3,[wojtek-t],"[Wojciech Tyczynski, Wojciech Tyczyński, wojtekt]",[wojtekt@google.com]
4,"[atlassian, ash2k]",[Mikhail Mazurskiy],"[126021+ash2k@users.noreply.github.com, mmazurskiy@atlassian.com, mmazurskiy@gitlab.com]"
5,[aditigupta96],[Aditi Gupta],[aditigpta@google.com]
6,[jm-franc],[Jean-Marc François],[jmfran@google.com]
7,[bartoszmajsak],[Bartosz Majsak],[bartosz.majsak@gmail.com]
8,[shajmakh],[Shereen Haj],[shajmakh@redhat.com]
9,[rkolchmeyer],[Robert Kolchmeyer],[rkolchmeyer@google.com]


# Shared accounts


In [19]:
print(len(shared), "shared logins")
print(sorted(shared))

27 shared logins
['apprenda', 'borg-land', 'cezaryzukowski', 'cwdsuzhou', 'datadog', 'denkensk', 'endocode', 'fujitsuenablingsoftwaretechnologygmbh', 'googlecloudplatform', 'hotelsdotcom', 'hpcloud', 'huawei-paas', 'ii', 'intelsdi-x', 'k82cn', 'kubernetes', 'linux-on-ibm-z', 'mesosphere', 'nilo19', 'nokia', 'nordix', 'pivotal-k8s', 'snowplayfire', 'vmware', 'zjj2wry', 'zju-sel', 'zlabjp']


In [20]:
# who is behind the DataDog fork?

dd = prs[prs.pr_login.str.lower() == "datadog"]
y = pr_commits[pr_commits.pr_number.isin(dd.pr_number)].merge(
    commits[["sha", "author_name", "author_email", "author_date"]], on="sha")
y.groupby(["author_name", "author_email"]).agg(
    commits=("sha", "size"), first=("author_date", "min"), last=("author_date", "max")).sort_values("first")

,,commits,first,last
author_name,author_email,,,
Laurent Bernaille,laurent.bernaille@datadoghq.com,19,2018-06-22 20:53:03+00:00,2020-07-09 17:08:17+00:00
Lars Ekman,lars.g.ekman@est.tech,2,2018-11-30 17:42:22+00:00,2018-12-11 09:07:04+00:00
Eric Mountain,eric.mountain@datadoghq.com,2,2019-11-04 09:52:19+00:00,2019-11-04 09:52:19+00:00
Nayef Ghattas,nayef.ghattas@datadoghq.com,2,2020-04-16 11:51:27+00:00,2021-01-08 16:39:29+00:00
Benjamin Pineau,benjamin.pineau@datadoghq.com,5,2020-05-15 18:17:57+00:00,2020-10-04 08:59:19+00:00
Julien Balestra,julien.balestra@datadoghq.com,1,2020-06-25 10:06:30+00:00,2020-06-25 10:06:30+00:00
Hemanth Malla,hemanth.malla@datadoghq.com,1,2020-11-17 22:12:20+00:00,2020-11-17 22:12:20+00:00
Baptiste Girard-Carrabin,baptiste.girardcarrabin@datadoghq.com,5,2025-03-04 14:29:42+00:00,2025-10-23 08:44:55+00:00
Valerian Roche,valerian.roche@datadoghq.com,1,2025-06-27 02:00:41+00:00,2025-06-27 02:00:41+00:00


In [21]:
# the two DataDog PRs merged in 2025 and their commit authors

dd25 = dd[(dd.merged_at >= start) & (dd.merged_at < end)]
z = pr_commits[pr_commits.pr_number.isin(dd25.pr_number)].merge(commits[["sha", "author_name", "author_email"]], on="sha")
z.groupby(["pr_number", "author_name", "author_email"]).size()

pr_number  author_name               author_email                         
131098     Baptiste Girard-Carrabin  baptiste.girardcarrabin@datadoghq.com    5
131799     Valerian Roche            valerian.roche@datadoghq.com             1
dtype: int64

# Borderline cases

In [22]:
# accounts whose commits are only merge commits, but that our measures do not flag as bots

merge_only = emails[(emails.n >= 10) & (emails.n_merge == emails.n) & ~emails.is_bot]
merge_only[["names", "n", "first", "last"]]

,names,n,first,last
author_email,,,,
joe@bedafamily.com,[Joe Beda],173,2014-09-09 20:50:06+00:00,2016-02-19 23:42:00+00:00
mbforbes@users.noreply.github.com,[Maxwell Forbes],62,2014-11-10 21:10:47+00:00,2015-06-26 02:37:04+00:00
spxtr@users.noreply.github.com,[Joe Finney],33,2015-12-11 17:54:32+00:00,2016-04-18 20:12:07+00:00


In [23]:
joe = commits[commits.author_email == "joe@bedafamily.com"]
print("all merge commits:", joe.is_merge.all())
joe.subject.head(3)

all merge commits: True


117099    Merge pull request #21142 from justinsb/install_git_package
136738     Merge pull request #3934 from justinsb/fix_etcd_extract_v2
136933             Merge pull request #3811 from jbeda/seqdiag-Docker
Name: subject, dtype: str

In [24]:
# most repetitive humans active in 2025 (at least 20 commits)

active25 = set(c25.author_email)
rep = emails[~emails.is_bot & (emails.n >= 20) & emails.index.isin(active25)].sort_values("template_ratio")
rep[["names", "n", "template_ratio", "share_via_pr"]].head(5)

,names,n,template_ratio,share_via_pr
author_email,,,,
ctadeu@gmail.com,"[Carlos Panato, Carlos Tadeu Panato Junior, cpanato]",139,0.561151,1.000000
haoweic@google.com,"[Haowei Cai, Haowei Cai (Roy)]",194,0.788660,0.994845
ppereira@microsoft.com,[Prince Pereira],29,0.827586,1.000000
bingzhe.sun@daocloud.io,[bzsuni],51,0.843137,1.000000
akiakiskyhand@gmail.com,[YamasouA],32,0.843750,1.000000


In [25]:
# cpanato: repeated messages, but every commit went through a PR

cp = commits[(commits.author_email == "ctadeu@gmail.com") & ~commits.is_merge]
print("commits:", len(cp), "| through a PR:", cp.via_pr.sum())
print(cp.template.value_counts().head(5))

commits: 139 | through a PR: 139
template
[go#] update to go#                                                            9
[go] bump images, dependencies and versions to go #                            8
bump images and versions to go # and distroless iptables                       6
bump images, dependencies and versions to go # and distroless iptables         6
bump dependencies, images and versions used to go # and distroless iptables    5
Name: count, dtype: int64


In [26]:
# cpanato is listed as a release manager in OWNERS_ALIASES (end of 2025)

snapshot = git("rev-list", "-1", "--before=2026-01-01T00:00:00Z", "master").strip()
aliases = git("show", f"{snapshot}:OWNERS_ALIASES")
block = aliases[aliases.index("release-managers:"):].split("\n\n")[0]
print(block[:400])

release-managers:
    - cpanato
    - jeremyrickard
    - justaugustus
    - palnabarun
    - puerco
    - saschagrunert
    - Verolop
    - xmudrii
  release-team-subproject-leads:
    - gracenng # Release Team subproject lead
    - katcosgrove # Release Team subproject lead
  build-image-approvers:
    - BenTheElder
    - cblecker
    - cpanato # SIG Technical Lead / RelEng subproject owner / Re


In [27]:
# save everything the next notebooks need

commits.drop(columns=["body"]).to_csv("data/commits.csv", index=False)
prs.to_csv("data/prs.csv", index=False)
pr_commits.to_csv("data/pr_commits.csv", index=False)
emails.reset_index().to_csv("data/email_stats.csv", index=False)
bots = sorted(["email:" + e for e in bot_emails] + ["login:" + l for l, b in login_is_bot.items() if b])
pd.concat([person_map.assign(is_bot=False),
           pd.DataFrame({"node": bots, "person_id": -1, "is_bot": True})]).to_csv("data/person_map.csv", index=False)
pd.Series(sorted(shared), name="login").to_csv("data/shared_logins.csv", index=False)
print("saved")

saved
